# Tensile and printability data — update with the first recycled samples (2026-11)

Newest Drive snapshot, `raw_data/11_02_26/` (pulled by `scripts/sync_drive.py`): cycle 0 unchanged, plus **40 new specimens at cycle 1** (`1x recycled`:
8 PP and HDPE formulations, 5 replicates each) and **corner warping** for 11 formulations. Extends `data_processing_v4.ipynb`.

Logic in `utils/instron.py`, `utils/analysis.py`, `utils/doe.py`; figures in `utils/plots.py`. Run in the `ml_gp_env` kernel. **Nothing is written by default.**

## Key points

* **Coverage:** 20 runs, 82 specimens analysed. All 12 formulations at cycle 0; 8 at cycle 1. **No TPU at cycle 1, nothing at cycles 2–3.**
* **One recycle loop shows no clear loss.** Average change: UTS +19 % (95 % CI −10 to +56 %, p = 0.19), modulus +6 %, strain +17 %, toughness +44 % (all p ≥ 0.18; 8 pairs).
* **The change depends on extrusion temperature** (exploratory, 8 points): for strain and toughness r = 0.93 and 0.91, and it survives dropping any one formulation (p ≤ 0.008). The GP agrees: temperature × cycle (Sobol 0.08–0.23) outweighs cycle alone (0.02–0.08).
* **F04 and F01 are the two to question.** F04 (180 °C) loses about 37 % of UTS; F01 doubles. Both also changed mean diameter, and the UTS and modulus changes follow the diameter change (r = −0.77, −0.82).
* **Thicker specimens read lower:** within a run, UTS and modulus fall with diameter (r = −0.49, −0.38). Diameter is a likely source of the replicate scatter.
* **Cycle 0 results stand:** polymer first, then PAL (significant for all four once each polymer has its own slope); polymer × PAL for modulus, strain, toughness; temperature not resolved.
* **Printability:** warping differs between formulations (p < 0.001), about 0.08–0.33 cm. TPU warps least; polymer p = 0.06. PAL: no sign of an effect. Temperature: no straight trend, a hint of a valley near 210 °C (p = 0.12). F09 is missing, F04 has one usable sample, and the sheet does not state the cycle.
* **DoE:** optimized LHS with recycling as an input, per polymer, conditioned on **all 20 runs**, **13 new runs** (PP 4, HDPE 4, TPU 5): PP and HDPE at cycles 2–3, TPU at 1–3. L2 (CD²) 0.201 → 0.044 over all runs, 0.443 → 0.084 within a polymer; worst within-polymer |r| 0.30 → 0.046. Cycles 2–3 stay thin (5 runs each). Plan: `doe/doe_plan_v5_conditioned.xlsx` (the 21-run version is `doe_plan_v4_conditioned.xlsx`). See section 9.
* **Same behaviour?** PAL keeps its direction after one loop; the temperature trend reverses (strain −4 → +13 %, toughness −13 → +22 % per +10 °C; exploratory).

**Caution:** cycle is confounded with test session, and batches differ more than replicates (lack of fit p < 0.001), so treat the recycle numbers as a first look.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "utils").exists() and (ROOT.parent / "utils").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from utils import config, io, doe, plots

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
plt.rcParams["figure.dpi"] = 110
p = config.paths(ROOT)

In [ ]:
from utils import instron, analysis as A

RAW = p.root / "raw_data" / "11_02_26"
design = pd.read_csv(p.doe / "doe_formulations.csv")
cat = instron.catalogue(io.extract_zip(RAW))
print((RAW / "CHANGES.txt").read_text()[:200])
cat.groupby("recycled_x").agg(specimens=("specimen_id", "size"), formulations=("formulation", "nunique"))

---
## 1 — Parse, set aside, screen

Same two set-asides as v4. No recycled specimen is removed; the pooled screen, now per run (formulation at one cycle), flags **f04r1_s4 on all four responses**: worth a question to the lab.

In [ ]:
SAVE_CSV = False                      # True -> write the per-specimen table to processed_data/tensile_doe/
CSV_NAME = "tensile_nov_summary"

curves, raw = instron.load_delivery(cat)
summary = instron.attach_doe(raw, design)
DROP = {"f11_s3": "failed test: no load recorded (UTS 0.12 MPa, modulus fit R² 0.001)",
        "f08_s1": "superseded by the f08 remake (f08-new): 119 % strain vs 7 % for the 3 remade specimens"}
kept, dropped = A.drop_specimens(summary, DROP)
kept0, kept1 = kept[kept["recycled_x"] == 0], kept[kept["recycled_x"] == 1]
if SAVE_CSV:
    summary.to_csv(io.save_path(p.tensile_doe, CSV_NAME, ext=".csv"), index=False)
print(f"{len(kept)} specimens kept: {len(kept0)} at cycle 0, {len(kept1)} at cycle 1")
dropped

In [ ]:
flags = A.pooled_outliers(kept)          # pooled log residuals from the run median, IQR fences
flags[flags["n_flags"] > 0]

In [ ]:
cv = A.cv_table(kept)
same8 = cv[cv["formulation"].isin(kept1["formulation"])]
print("median CV %, all runs:"); print(cv.groupby("cycle")[list(A.RESPONSES)].median().round(1))
print("median CV %, the 8 formulations with both cycles:"); print(same8.groupby("cycle")[list(A.RESPONSES)].median().round(1))
cv.round(2)

---
## 2 — Curves and properties

Cycle 0 figures as in v4; the properties-vs-factors figure now also shows cycle 1 (hollow).

In [ ]:
SAVE_FIG = False                      # True -> write the figure to processed_data/figures/
FIG_NAME = "tensile_nov_curves_cycle0"

plots.tensile_curves_by_polymer(curves, kept0, save=io.save_path(p.figures, FIG_NAME, SAVE_FIG, ".png"))

In [ ]:
SAVE_FIG = False                      # True -> write the figure to processed_data/figures/
FIG_NAME = "tensile_nov_props_cycle0"

plots.props_by_formulation_log(kept0, save=io.save_path(p.figures, FIG_NAME, SAVE_FIG, ".png"))

In [ ]:
SAVE_FIG = False                      # True -> write the figure to processed_data/figures/
FIG_NAME = "tensile_nov_props_vs_factors"

plots.props_vs_factors_log(kept, save=io.save_path(p.figures, FIG_NAME, SAVE_FIG, ".png"))

---
## 3 — The recycle step: the same formulation at cycles 0 and 1

The formulation is the unit: 8 pairs. Ratio > 1 means higher after recycling.

**Gain or loss?** Each thin line is one formulation; the black line is the average of the 8, with its 95 % CI. Cycle 0 = first extrusion (lab's 0x), cycle 1 = one loop (1x).

In [ ]:
pairs = A.cycle_pairs(kept)
effect = A.cycle_effect(pairs)
means = A.cycle_means(pairs)
means.round(3)

In [ ]:
SAVE_FIG = False                      # True -> write the figure to processed_data/figures/
FIG_NAME = "tensile_nov_cycle_gain_loss"

plots.cycle_gain_loss(pairs, effect, means, save=io.save_path(p.figures, FIG_NAME, SAVE_FIG, ".png"))

In [ ]:
SAVE_FIG = False                      # True -> write the figure to processed_data/figures/
FIG_NAME = "tensile_nov_curves_by_cycle"

plots.curves_by_cycle(curves, kept, save=io.save_path(p.figures, FIG_NAME, SAVE_FIG, ".png"))

In [ ]:
effect.round(3)

In [ ]:
SAVE_FIG = False                      # True -> write the figure to processed_data/figures/
FIG_NAME = "tensile_nov_cycle_forest"

plots.cycle_ratio_forest(pairs, effect, save=io.save_path(p.figures, FIG_NAME, SAVE_FIG, ".png"))

**Is the behaviour the same?** Slopes per cycle on the 8 formulations with both, % per +10 PAL points and per +10 °C, each adjusted for the other input; `p_change` tests whether the slope changed. PAL keeps its direction; temperature reverses for UTS, strain and toughness. Exploratory.

In [ ]:
A.cycle_trend_table(kept).round(3)

In [ ]:
SAVE_FIG = False                      # True -> write the figure to processed_data/figures/
FIG_NAME = "tensile_nov_cycle_trends"

plots.cycle_trends(kept, save=io.save_path(p.figures, FIG_NAME, SAVE_FIG, ".png"))

Does the change depend on the factors? Temperature and diameter stand out. **Exploratory:** 8 points, several tests, chosen after seeing the data. `p_loo` is the largest p after removing any one formulation.

In [ ]:
A.cycle_interactions(pairs).round(3)

In [ ]:
A.cycle_vs_factors(pairs, kept).round(3)

In [ ]:
SAVE_FIG = False                      # True -> write the figure to processed_data/figures/
FIG_NAME = "tensile_nov_cycle_vs_temperature"

plots.cycle_ratio_vs_temperature(pairs, save=io.save_path(p.figures, FIG_NAME, SAVE_FIG, ".png"))

Thicker specimens of one run read lower UTS and modulus. A slope of −2 would be pure area (stress ∝ 1/d²), so this could be measurement of the cross-section, or voids in thick filament.

In [ ]:
A.diameter_effect(kept).round(3)

In [ ]:
# Sensitivity: without the flagged specimen, and without F01
print(A.cycle_effect(A.cycle_pairs(kept[kept["specimen_id"] != "f04r1_s4"])).query("group == 'all'")[["response", "ratio", "lo", "hi", "p"]].round(3))
print(A.cycle_effect(pairs[pairs["formulation"] != 1]).query("group == 'all'")[["response", "ratio", "lo", "hi", "p"]].round(3))

---
## 4 — Pair plot and correlation between the responses

In [ ]:
SAVE_FIG = False                      # True -> write the figure to processed_data/figures/
FIG_NAME = "tensile_nov_pairs"

plots.pair_plot_specimens(kept, save=io.save_path(p.figures, FIG_NAME, SAVE_FIG, ".png"))

In [ ]:
corrs = A.response_correlation(kept, polymers=config.DOE.polymers)
plots.response_correlation(corrs)

---
## 5 — ANOVA

Pooling all 20 run means would treat the 8 repeated formulations as independent. So: **polymer, PAL and temperature are tested on the 12 cycle-0 means** and **the recycle step on the 8 pairs**; a mixed model is a cross-check.

In [ ]:
A.lack_of_fit(kept, rhs="C(polymer) * pal + temp + cycle").round(4)

In [ ]:
an0 = A.anova_means(kept0, rhs="C(polymer) * pal + temp")   # one PAL slope per polymer
x = np.log(pairs.assign(_r=pairs["ratio"]).pivot(index="formulation", columns="response", values="_r"))
t = x.mean() / (x.std(ddof=1) / np.sqrt(len(x)))
step = pd.DataFrame({"response": t.index, "effect": "recycle step (cycle 0 → 1)", "df": 1, "F": t.to_numpy() ** 2,
                     "p": effect.query("group == 'all'").set_index("response").loc[t.index, "p"].to_numpy(),
                     "partial_eta2": (t ** 2 / (t ** 2 + len(x) - 1)).to_numpy()})
an = pd.concat([an0, step], ignore_index=True)
an.round(4)

In [ ]:
SAVE_FIG = False                      # True -> write the figure to processed_data/figures/
FIG_NAME = "tensile_nov_anova"

plots.anova_effects(an, save=io.save_path(p.figures, FIG_NAME, SAVE_FIG, ".png"))

In [ ]:
A.interaction_tests(kept0).round(4)

In [ ]:
print("PAL effect, % per +10 PAL points, cycle 0"); print(A.pal_effect_by_polymer(kept0).round(1))
print("cycle 1"); print(A.pal_effect_by_polymer(kept1).round(1))

In [ ]:
A.anova_means(kept1, rhs="C(polymer) + pal + temp").round(4)   # the 8 cycle-1 means

In [ ]:
import statsmodels.formula.api as smf
import warnings
rows = []
for r in A.RESPONSES:                    # cross-check of the recycle step: random intercept per formulation
    fm = A._formulation_means(kept, r)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        m = smf.mixedlm("ly ~ C(polymer) * pal + temp + cycle", fm, groups=fm["formulation"]).fit(reml=True)
    rows.append({"response": r, "ratio": np.exp(m.params["cycle"]), "p": m.pvalues["cycle"]})
pd.DataFrame(rows).round(3)

---
## 6 — Partial dependence and Sobol indices with the recycle cycle as an input

GP as in v4 with the cycle as one more input. Each polymer is drawn only at cycles it has data for. Sobol indices use PP and HDPE only: TPU has no recycled data, and is not extrapolated. About 5 min.

In [ ]:
pds = {r: A.gp_partial_dependence_cycles(kept, r, config.DOE.polymers) for r in A.RESPONSES}
{r: d.attrs["kernel"] for r, d in pds.items()}

In [ ]:
SAVE_FIG = False                      # True -> write the figure to processed_data/figures/
FIG_NAME = "tensile_nov_partial_dependence"

plots.partial_dependence_cycles(pds, save=io.save_path(p.figures, FIG_NAME, SAVE_FIG, ".png"))

In [ ]:
sob = pd.concat([A.gp_sobol_cycle(kept, r) for r in A.RESPONSES], ignore_index=True)
(sob.assign(order=sob["order"].astype(str)).pivot_table(index=["order", "term"], columns="response", values="median", sort=False)
 [list(A.RESPONSES)].round(3))

---
## 7 — Printability: corner warping

From the lab's `Sample Warping.xlsx`. Counts of 4, 8 and 12 suggest 4 corners on 1, 2 or 3 samples (**to confirm**). F09 is missing, F04 has one usable sample, and **the sheet does not say which cycle**, so it is compared with cycle 0 here. Readings are in steps of 0.05–0.1 cm.

In [ ]:
warp = A.load_warping(RAW / "extracted" / "printability" / "Sample Warping.xlsx", design)
A.warping_summary(warp).round(3)

In [ ]:
SAVE_FIG = False                      # True -> write the figure to processed_data/figures/
FIG_NAME = "warping_by_formulation"

plots.warping_by_formulation(warp, save=io.save_path(p.figures, FIG_NAME, SAVE_FIG, ".png"))

In [ ]:
wt = A.warping_tests(warp, kept)
print("between formulations (Kruskal-Wallis):", {k: round(float(v), 6) for k, v in wt["between"].items()})
print(wt["anova"].round(4))
wt["tensile"].round(3)

**Does adding PAL or changing temperature move warping?** Straight-line slopes in cm, and a curvature test for temperature. The LVGP surfaces and its leave-one-out check are in `modeling/lvgp_recycling.ipynb`.

In [ ]:
A.warping_slopes(warp).round(3)

In [ ]:
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm
ws = A.warping_summary(warp).rename(columns={"polymer_system": "polymer", "pal_pct": "pal", "extr_temp_C": "temp"})
ws["t"] = ws["temp"] - ws["temp"].mean()
straight, curved = smf.ols("mean ~ C(polymer) + pal + t", ws).fit(), smf.ols("mean ~ C(polymer) + pal + t + I(t ** 2)", ws).fit()
print("lowest warping at about", round(ws["temp"].mean() - curved.params["t"] / (2 * curved.params["I(t ** 2)"])), "degC")
anova_lm(straight, curved).round(4)

Warping rises with UTS and modulus across formulations (Spearman 0.75, 0.80), but that is mostly the polymer: TPU is soft and warps least. With 4 formulations per polymer it cannot be separated within a polymer yet.

---
## 8 — Trade-offs

In [ ]:
SAVE_FIG = False                      # True -> write the figure to processed_data/figures/
FIG_NAME = "tensile_nov_tradeoffs"

plots.tradeoffs(kept, save=io.save_path(p.figures, FIG_NAME, SAVE_FIG, ".png"))

---
## 9 — What this means for the DoE

Keep the LHS with the recycle cycle as an input, **conditioned on all 20 runs**. Compared here with the old plan unchanged and the standard baselines; the reasoning is in the deck. Formulations at two cycles share one PAL / temperature point, so they take one band. Takes a few minutes.

In [ ]:
fixed = doe.existing_runs(design, cat)
fixed.groupby(["polymer_system", "recycle_cycle"]).size().unstack(fill_value=0)

In [ ]:
cols = ["polymer_system", "PAL_pct", "extrusion_temp_C", "recycle_cycle"]
v3 = pd.read_excel(p.doe / "doe_plan_v3_conditioned.xlsx", sheet_name="configurations")
union_v3 = pd.concat([fixed[cols], v3[cols]], ignore_index=True)
print("October plan unchanged, plus the 20 runs:"); print(union_v3.groupby(["polymer_system", "recycle_cycle"]).size().unstack(fill_value=0))
pd.Series(doe.design_quality(union_v3)).to_frame("v3 as is")

In [ ]:
SPLIT = {"PP+PAL": 4, "HDPE+PAL": 4, "TPU+PAL": 5}     # 13 new runs; TPU gets the extra one (no recycled data yet)
N_NEW = sum(SPLIT.values())
seeds = doe.conditioned_seeds(fixed, range(10), n_per_polymer=SPLIT)
seeds.sort_values(["poly_fill_dist", "all_min_dist"], ascending=[True, False]).round(3)

In [ ]:
base = doe.compare_conditioned_baselines(fixed, n_seeds=10, n_new=N_NEW)
base.groupby("method").mean(numeric_only=True).round(3).T

In [ ]:
SAVE_PLAN = False                     # True -> write doe/<PLAN_NAME>.xlsx and .csv (already written once, 2026-11-02)
PLAN_NAME = "doe_plan_v5_conditioned"   # 13 new runs; doe_plan_v4_conditioned = the 21-run version
PICK_SEED = 8                         # seeds tie on discrepancy; this one has the smallest within-polymer gap and the largest closest-pair distance

new, q = doe.conditioned_sliced_lhs(fixed, SPLIT, seed=PICK_SEED)
if SAVE_PLAN:
    doe.write_plan_workbook(new, fixed, p.doe / f"{PLAN_NAME}.xlsx")
    new.assign(tensile_replicates=5).to_csv(p.doe / f"{PLAN_NAME}.csv", index=False)
union = pd.concat([fixed[cols], new[cols]], ignore_index=True)
print(pd.Series(doe.design_quality(fixed[cols]), name="20 fabricated").to_frame().join(pd.Series(doe.design_quality(union), name=f"+ {len(new)} new")))
new

In [ ]:
SAVE_FIG = False                      # True -> write the figure to processed_data/figures/
FIG_NAME = "doe_v5_design_space"

plots.doe_design_space(new, existing=fixed, color_by="cycle", save=io.save_path(p.figures, FIG_NAME, SAVE_FIG, ".png"))

In [ ]:
SAVE_FIG = False                      # True -> write the figure to processed_data/figures/
FIG_NAME = "doe_v5_input_correlation"

plots.doe_correlation(union, title="Correlation between inputs: 20 fabricated + 13 new runs", save=io.save_path(p.figures, FIG_NAME, SAVE_FIG, ".png"))

**Balance.** Runs per polymer × cycle before and after, and each polymer's PAL and temperature settings. With 13 runs, cycles 2–3 stay thin (5 runs each, TPU one each).

In [ ]:
SAVE_FIG = False                      # True -> write the figure to processed_data/figures/
FIG_NAME = "doe_v5_balance"

print(union.groupby(["polymer_system", "recycle_cycle"]).size().unstack(fill_value=0))
plots.design_balance(new, fixed, save=io.save_path(p.figures, FIG_NAME, SAVE_FIG, ".png"))

**Compared with repeating the same formulations.** The original plan took the 12 formulations to 1×, 5×, 10× (0, 4, 9 loops); the middle column is the same idea over cycles 0–3. Loops are given both ways: one recycled batch gives all 5 specimens of a configuration, or each replicate is recycled separately (to confirm with the lab). Repeating needs no compounding and gives direct pairs at every cycle; this plan gives 11 PAL–temperature points per polymer instead of 4, and independent inputs.

In [ ]:
rep = pd.concat([design.assign(recycle_cycle=c) for c in range(4)], ignore_index=True)[cols]     # the same 12 at cycles 0-3
n_done = len(fixed)
pd.DataFrame({"original plan": {"loops tested": "0, 4, 9", "PAL-T points per polymer": 4, "configurations": 36, "new specimens": 24 * 5, "loops, batch per config": 12 * 9, "loops, batch per replicate": 12 * 9 * 5, "to compound": 0},
              "same 12, cycles 0-3": {"loops tested": "0, 1, 2, 3", "PAL-T points per polymer": 4, "configurations": len(rep), "new specimens": (len(rep) - n_done) * 5, "loops, batch per config": 12 * 3, "loops, batch per replicate": 12 * 3 * 5, "to compound": 0},
              "this plan": {"loops tested": "0, 1, 2, 3", "PAL-T points per polymer": 11, "configurations": len(union), "new specimens": len(new) * 5, "loops, batch per config": int(new["recycle_cycle"].sum() + fixed["recycle_cycle"].sum()), "loops, batch per replicate": 5 * int(new["recycle_cycle"].sum() + fixed["recycle_cycle"].sum()), "to compound": len(new)}})

In [ ]:
pd.DataFrame({"same 12, cycles 0-3": doe.design_quality(rep), "this plan": doe.design_quality(union)})   # same metrics; note the fill distance favours the larger repeated plan

---
## Open questions for the lab

1. Is TPU at cycle 1 made or planned? If made, it joins the fixed runs and the design changes.
2. Which cycle are the warping samples? What is one reading: 4 corners per sample? Is F09 coming?
3. F04 at cycle 1, especially f04r1_s4, and F01 at cycle 0: anything different in the batch or test session?
4. How is the diameter measured, and is the second extrusion at the same temperature as the first?